# TOP ошибок и выборка ИНН по ФП/СФП

Аналитика построена по логике отчета `analysis_scenario_ipu_results.ipynb`:
- дедупликация карточек по двум потокам (scenario / ipu),
- выделение ошибок на этапе сценариев и ИПУ,
- TOP-10 по ошибкам в разрезе сегментов,
- выборка до 100 уникальных ИНН для TOP-5 в каждом сегменте,
- экспорт в Excel (каждая секция на отдельном листе).

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
# Источники и период выровнены с analysis_fp_1_2_two_report_logics.ipynb,
# чтобы итоговые цифры совпадали между тетрадками.
ALLOWED_SOURCES = {
    "Н2.0", "Справочник CRM-системы", "CRM-система",
}
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]
SCR_DATE_COLS = ["END_DATE_SCR_FCT", "END_DATE_SCR_PLAN"]
IPU_DATE_COLS = ["FIRST_END_DATE_EVENT", "NEW_PLAN_END_DATE_EVT", "END_EVENT_DATE_FACT"]

SCR_ERROR_TOKENS = [
    "информация проверена",
    "отсутствуют основания для выявления",
]
IPU_ERROR_TOKENS = [
    "срок окончания плана истек",
    "результат реализации по состоянию",
    "отсутствует",
]


_LAT2CYR = str.maketrans({
    "A": "А", "B": "В", "C": "С", "E": "Е", "H": "Н", "K": "К",
    "M": "М", "O": "О", "P": "Р", "T": "Т", "X": "Х",
    "a": "а", "c": "с", "e": "е", "o": "о", "p": "р", "x": "х", "y": "у",
})


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


def norm_text(value):
    if pd.isna(value) or value == "":
        return ""
    s = str(value)
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(_LAT2CYR)
    s = re.sub(r"[\s\xa0\u200b\ufeff]+", " ", s)
    return s.strip()


def normalize_source(value: str) -> str:
    if pd.isna(value):
        return ""
    src = norm_text(value).upper()
    return src.replace(" ", "")


def has_all_tokens(text: str, tokens: list[str]) -> bool:
    base = norm_text(text).lower()
    return all(token in base for token in tokens)


def is_scenario_error(text: str) -> bool:
    return has_all_tokens(text, SCR_ERROR_TOKENS)


def is_ipu_error(text: str) -> bool:
    return has_all_tokens(text, IPU_ERROR_TOKENS)


# Жесткие банковые пути (без fallback).
DATA_DIR = Path('/home/jovyan/documents/DMUKP_FP_DEF/data')
CRM_FILE = DATA_DIR / 'crm_last_version.csv'
EXCEL_PATH = DATA_DIR / 'error_top_inn_by_segment.xlsx'

print(f"CRM_FILE: {CRM_FILE}")
print(f"EXCEL_PATH: {EXCEL_PATH}")

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
if len(raw.columns) == 1 and ";" in str(raw.columns[0]):
    raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False, sep=";")
raw.columns = raw.columns.str.strip()

if "DESC_TEXT.1" in raw.columns and "DESC_TEXT_1" not in raw.columns:
    raw = raw.rename(columns={"DESC_TEXT.1": "DESC_TEXT_1"})

if "VAL" in raw.columns:
    source_norm = raw["VAL"].apply(normalize_source)
    allowed_norm = {normalize_source(v) for v in ALLOWED_SOURCES}
    filtered = raw[source_norm.isin(allowed_norm)].copy()
    # Если в локальной выгрузке источники отличаются от отчета, не обнуляем выборку.
    raw = filtered if len(filtered) > 0 else raw.copy()

# Поддержка как «сырых» CRM-колонок, так и уже подготовленного набора.
inn_col = "X_INN" if "X_INN" in raw.columns else "inn"
segment_col = "X_AREA_RESP" if "X_AREA_RESP" in raw.columns else "segment"
fp_num_col = "NUMBER_FP_SFP" if "NUMBER_FP_SFP" in raw.columns else "fp_num"
fp_type_col = "TYPE_FP" if "TYPE_FP" in raw.columns else "fp_type"

raw["inn"] = raw[inn_col].apply(normalize_inn)
raw["dt"] = pd.to_datetime(raw["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
raw = raw[(raw["dt"] >= DATE_FROM) & (raw["dt"] <= DATE_TO)].copy()

# Анализируем только 3 целевых сегмента; прочие (в т.ч. ДРПА) исключаем.
raw["segment"] = raw[segment_col].astype(str).str.strip().map(SEGMENT_MAP)
raw["fp_num"] = raw[fp_num_col].astype(str).str.strip()
raw["fp_type"] = raw[fp_type_col].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})

raw = raw.dropna(subset=["inn", "fp_num"]).copy()
raw = raw[raw["segment"].notna()].copy()

raw["scenario"] = raw["DESC_TEXT_1"].apply(norm_text)
raw["ipu"] = raw["DESC_TEXT"].apply(norm_text)

for col in SCR_DATE_COLS + IPU_DATE_COLS:
    raw[f"_{col}_dt"] = pd.to_datetime(raw[col], dayfirst=True, errors="coerce")

scr_dt_cols = [f"_{col}_dt" for col in SCR_DATE_COLS]
ipu_dt_cols = [f"_{col}_dt" for col in IPU_DATE_COLS]

raw["_scr_max"] = raw[scr_dt_cols].max(axis=1)
raw["_ipu_max"] = raw[ipu_dt_cols].max(axis=1)

# Делим на потоки по наличию дат ИПУ внутри карточки.
grp_has_ipu = raw.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()
stream1_raw = raw[~grp_has_ipu].copy()
stream2_raw = raw[grp_has_ipu].copy()

stream1_raw = stream1_raw[stream1_raw.groupby(DEDUP_KEY)["_scr_max"].transform("max").notna()]
stream1_sorted = stream1_raw.sort_values(["_END_DATE_SCR_FCT_dt", "_END_DATE_SCR_PLAN_dt"], ascending=[False, False], na_position="last")
df_stream1 = stream1_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream1["stream"] = "scenario"

stream2_raw = stream2_raw[stream2_raw.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()]
stream2_sorted = stream2_raw.sort_values(["_END_EVENT_DATE_FACT_dt", "_NEW_PLAN_END_DATE_EVT_dt", "_FIRST_END_DATE_EVENT_dt"], ascending=[False, False, False], na_position="last")
df_stream2 = stream2_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream2["stream"] = "ipu"

df = pd.concat([df_stream1, df_stream2], ignore_index=True)

# Выделяем ошибки по текстовым токенам (устойчиво к префиксам/вариантам формулировок).
df["scr_class"] = df["scenario"].apply(lambda x: "ошибка" if is_scenario_error(x) else "прочее")
df["ipu_class"] = df["ipu"].apply(lambda x: "ошибка" if is_ipu_error(x) else "прочее")

print(f"raw rows: {len(raw):,}")
print(f"dedup rows (df): {len(df):,}")
print(df[["segment", "fp_num", "fp_type", "stream", "scr_class", "ipu_class"]].head())

In [ ]:
# Эталонная база total_count: как в logic_no_stream_split (без stream-разделения).
base_cards = raw.drop_duplicates(DEDUP_KEY).copy()
base_totals = (
    base_cards
    .groupby(["segment", "fp_num", "fp_type"], as_index=False)
    .size()
    .rename(columns={"size": "total_count"})
)


def top_by_segment_error(stage_df: pd.DataFrame, class_col: str, totals_df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    error_rows = stage_df[stage_df[class_col] == "ошибка"].copy()
    if error_rows.empty:
        return pd.DataFrame(
            columns=["segment", "rank", "fp_num", "fp_type", "error_count", "total_count", "error_share_pct"]
        )

    error_agg = (
        error_rows
        .groupby(["segment", "fp_num", "fp_type"], as_index=False)
        .size()
        .rename(columns={"size": "error_count"})
    )

    agg = error_agg.merge(totals_df, on=["segment", "fp_num", "fp_type"], how="left")
    agg["total_count"] = agg["total_count"].fillna(0).astype(int)
    agg["error_share_pct"] = (
        (agg["error_count"] / agg["total_count"].where(agg["total_count"] > 0, pd.NA)) * 100
    ).fillna(0).round(2)

    agg = agg.sort_values(["segment", "error_count", "fp_num", "fp_type"], ascending=[True, False, True, True])
    agg["rank"] = agg.groupby("segment").cumcount() + 1
    return agg[agg["rank"] <= top_n].reset_index(drop=True)


scenario_stage = df.copy()
ipu_stage = df[df["stream"] == "ipu"].copy()

top10_scenario_error = top_by_segment_error(scenario_stage, class_col="scr_class", totals_df=base_totals, top_n=10)
top10_ipu_error = top_by_segment_error(ipu_stage, class_col="ipu_class", totals_df=base_totals, top_n=10)

# Контроль сверки для ФП 1.2 в ККБ с comparison-ноутбуком.
control_fp12_kkb_total = base_totals.loc[
    (base_totals["segment"] == "ККБ")
    & (base_totals["fp_num"] == "1.2")
    & (base_totals["fp_type"] == "ФП"),
    "total_count"
]
control_fp12_kkb_total = int(control_fp12_kkb_total.iloc[0]) if len(control_fp12_kkb_total) else 0

print("TOP-10 по ошибкам сценариев")
display(top10_scenario_error)
print("TOP-10 по ошибкам ИПУ")
display(top10_ipu_error)
print("Контроль total_count для ФП 1.2 (ККБ, no_stream_split):", control_fp12_kkb_total)

In [ ]:
def top5_inn_lists(stage_df: pd.DataFrame, top_df: pd.DataFrame, class_col: str, stage_name: str, inn_limit: int = 100) -> pd.DataFrame:
    top5 = top_df[top_df["rank"] <= 5].copy()
    rows = []

    for item in top5.itertuples(index=False):
        sub = stage_df[
            (stage_df["segment"] == item.segment)
            & (stage_df["fp_num"] == item.fp_num)
            & (stage_df["fp_type"] == item.fp_type)
            & (stage_df[class_col] == "ошибка")
        ].copy()

        inn_sample = sub["inn"].dropna().drop_duplicates().head(inn_limit)
        for idx, inn in enumerate(inn_sample, start=1):
            rows.append(
                {
                    "analysis_stage": stage_name,
                    "segment": item.segment,
                    "rank": int(item.rank),
                    "fp_num": item.fp_num,
                    "fp_type": item.fp_type,
                    "error_count": int(item.error_count),
                    "total_count": int(item.total_count),
                    "error_share_pct": float(item.error_share_pct),
                    "inn_position": idx,
                    "inn": inn,
                }
            )

    return pd.DataFrame(rows)


top5_scenario_inn100 = top5_inn_lists(
    stage_df=scenario_stage,
    top_df=top10_scenario_error,
    class_col="scr_class",
    stage_name="scenario",
    inn_limit=100,
)

top5_ipu_inn100 = top5_inn_lists(
    stage_df=ipu_stage,
    top_df=top10_ipu_error,
    class_col="ipu_class",
    stage_name="ipu",
    inn_limit=100,
)

print("ИНН по TOP-5 ошибок сценариев")
display(top5_scenario_inn100.head(20))
print("ИНН по TOP-5 ошибок ИПУ")
display(top5_ipu_inn100.head(20))

## Групповой анализ ФП/СФП и дефолты (логика ДР)

Добавлены проверки по группам:
1. `14.3*`
2. `15.2*`
3. `16.1*`
4. `43.1У`, `43.2У`, `43.3У`, `43.4У`

Для каждой группы считаются:
- абсолютные ошибки (`scenario`, `ipu`, `any_stage`),
- относительные ошибки к общему числу выявлений,
- количество ИНН с такими ФП/СФП и доля ИНН, пришедших к дефолту по DR-логике (`dataset_fp_default.pkl`).

In [ ]:
def group_mask(fp_series: pd.Series, group_key: str) -> pd.Series:
    s = fp_series.astype(str).str.strip().str.upper()
    if group_key == "grp_14_3":
        return s.str.match(r"^14\.3")
    if group_key == "grp_15_2":
        return s.str.match(r"^15\.2")
    if group_key == "grp_16_1":
        return s.str.match(r"^16\.1")
    if group_key == "grp_43_u":
        return s.isin({"43.1У", "43.2У", "43.3У", "43.4У"})
    return pd.Series(False, index=fp_series.index)


group_specs = [
    {"group_key": "grp_14_3", "group_label": "14.3*"},
    {"group_key": "grp_15_2", "group_label": "15.2*"},
    {"group_key": "grp_16_1", "group_label": "16.1*"},
    {"group_key": "grp_43_u", "group_label": "43.1У/43.2У/43.3У/43.4У"},
]


def aggregate_group_error_stats(group_key: str, group_label: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    base_sub = base_cards[group_mask(base_cards["fp_num"], group_key)].copy()
    total_by_segment = (
        base_sub.groupby("segment", as_index=False)
        .size()
        .rename(columns={"size": "total_count"})
    )

    scenario_err = scenario_stage[(scenario_stage["scr_class"] == "ошибка") & group_mask(scenario_stage["fp_num"], group_key)].copy()
    scenario_err = scenario_err.drop_duplicates(DEDUP_KEY)
    scenario_by_segment = (
        scenario_err.groupby("segment", as_index=False)
        .size()
        .rename(columns={"size": "scenario_error_count"})
    )

    ipu_err = ipu_stage[(ipu_stage["ipu_class"] == "ошибка") & group_mask(ipu_stage["fp_num"], group_key)].copy()
    ipu_err = ipu_err.drop_duplicates(DEDUP_KEY)
    ipu_by_segment = (
        ipu_err.groupby("segment", as_index=False)
        .size()
        .rename(columns={"size": "ipu_error_count"})
    )

    any_err_keys = pd.concat([
        scenario_err[DEDUP_KEY],
        ipu_err[DEDUP_KEY],
    ], ignore_index=True).drop_duplicates()

    any_err_cards = base_sub.merge(any_err_keys, on=DEDUP_KEY, how="inner")
    any_by_segment = (
        any_err_cards.groupby("segment", as_index=False)
        .size()
        .rename(columns={"size": "any_stage_error_count"})
    )

    seg = (
        total_by_segment
        .merge(scenario_by_segment, on="segment", how="left")
        .merge(ipu_by_segment, on="segment", how="left")
        .merge(any_by_segment, on="segment", how="left")
        .fillna(0)
    )

    for c in ["total_count", "scenario_error_count", "ipu_error_count", "any_stage_error_count"]:
        seg[c] = seg[c].astype(int)

    seg["scenario_error_share_pct"] = (seg["scenario_error_count"] / seg["total_count"].where(seg["total_count"] > 0, pd.NA) * 100).fillna(0).round(2)
    seg["ipu_error_share_pct"] = (seg["ipu_error_count"] / seg["total_count"].where(seg["total_count"] > 0, pd.NA) * 100).fillna(0).round(2)
    seg["any_stage_error_share_pct"] = (seg["any_stage_error_count"] / seg["total_count"].where(seg["total_count"] > 0, pd.NA) * 100).fillna(0).round(2)
    seg.insert(0, "group_label", group_label)

    total_row = pd.DataFrame([{
        "group_label": group_label,
        "segment": "ИТОГО",
        "total_count": int(seg["total_count"].sum()),
        "scenario_error_count": int(seg["scenario_error_count"].sum()),
        "ipu_error_count": int(seg["ipu_error_count"].sum()),
        "any_stage_error_count": int(seg["any_stage_error_count"].sum()),
    }])
    for c_num, c_share in [
        ("scenario_error_count", "scenario_error_share_pct"),
        ("ipu_error_count", "ipu_error_share_pct"),
        ("any_stage_error_count", "any_stage_error_share_pct"),
    ]:
        total_row[c_share] = (total_row[c_num] / total_row["total_count"].where(total_row["total_count"] > 0, pd.NA) * 100).fillna(0).round(2)

    seg_with_total = pd.concat([seg, total_row], ignore_index=True)

    return seg_with_total, base_sub


group_error_stats_parts = []
group_card_subsets = {}
for spec in group_specs:
    part, subset = aggregate_group_error_stats(spec["group_key"], spec["group_label"])
    group_error_stats_parts.append(part)
    group_card_subsets[spec["group_label"]] = subset

group_error_stats = pd.concat(group_error_stats_parts, ignore_index=True)

print("Групповой анализ ошибок (абсолютные и относительные):")
display(group_error_stats)

In [ ]:
# Дефолты по логике ДР: используем итоговый датасет build_dataset (dataset_fp_default.pkl).
DATASET_FP_DEFAULT_FILE = DATA_DIR / "dataset_fp_default.pkl"

dataset_fp_default = pd.read_pickle(DATASET_FP_DEFAULT_FILE)
dataset_fp_default.columns = [str(c).strip() for c in dataset_fp_default.columns]

if "inn" not in dataset_fp_default.columns or "default_flag" not in dataset_fp_default.columns:
    raise KeyError("В dataset_fp_default.pkl отсутствуют колонки 'inn' или 'default_flag'.")

dataset_fp_default["inn"] = dataset_fp_default["inn"].apply(normalize_inn)
default_inn_set = set(dataset_fp_default.loc[dataset_fp_default["default_flag"].fillna(0).astype(int) == 1, "inn"].dropna().unique())


def build_default_stats_for_group(group_label: str, base_sub: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    inn_seg = (
        base_sub.groupby(["segment", "inn"], as_index=False)
        .size()
        .rename(columns={"size": "cards_n"})
    )
    inn_seg["default_flag_dr"] = inn_seg["inn"].isin(default_inn_set).astype(int)

    by_segment = (
        inn_seg.groupby("segment", as_index=False)
        .agg(
            inn_total=("inn", "nunique"),
            inn_default_dr=("default_flag_dr", "sum"),
        )
        .sort_values("inn_total", ascending=False)
    )
    by_segment["default_rate_dr_pct"] = (by_segment["inn_default_dr"] / by_segment["inn_total"].where(by_segment["inn_total"] > 0, pd.NA) * 100).fillna(0).round(2)
    by_segment.insert(0, "group_label", group_label)

    total = pd.DataFrame([{
        "group_label": group_label,
        "segment": "ИТОГО",
        "inn_total": int(by_segment["inn_total"].sum()),
        "inn_default_dr": int(by_segment["inn_default_dr"].sum()),
    }])
    total["default_rate_dr_pct"] = (total["inn_default_dr"] / total["inn_total"].where(total["inn_total"] > 0, pd.NA) * 100).fillna(0).round(2)

    by_segment_full = pd.concat([by_segment, total], ignore_index=True)

    return by_segment_full, inn_seg


group_default_stats_parts = []
group_default_inn_details_parts = []
for spec in group_specs:
    label = spec["group_label"]
    stats_part, details_part = build_default_stats_for_group(label, group_card_subsets[label])
    group_default_stats_parts.append(stats_part)
    details_part.insert(0, "group_label", label)
    group_default_inn_details_parts.append(details_part)

group_default_stats = pd.concat(group_default_stats_parts, ignore_index=True)
group_default_inn_details = pd.concat(group_default_inn_details_parts, ignore_index=True)

print("ИНН с выявленными ФП/СФП из групп 1-4, пришедшие к дефолту (логика ДР):")
display(group_default_stats)
print("Детализация по ИНН:")
display(group_default_inn_details.head(200))

In [ ]:
# Контрольные проверки для новых секций.
new_checks = pd.DataFrame([
    {
        "check_name": "group_error_stats_has_rows",
        "passed": bool(len(group_error_stats) > 0),
    },
    {
        "check_name": "group_default_stats_has_rows",
        "passed": bool(len(group_default_stats) > 0),
    },
    {
        "check_name": "group_default_rate_in_range",
        "passed": bool(((group_default_stats["default_rate_dr_pct"] >= 0) & (group_default_stats["default_rate_dr_pct"] <= 100)).all()) if len(group_default_stats) else True,
    },
])

print("Проверки новых секций:")
display(new_checks)

In [ ]:
# Расширенный экспорт с новыми групповыми секциями.
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    top10_scenario_error.to_excel(writer, sheet_name="TOP10_Scenario_Error", index=False)
    top10_ipu_error.to_excel(writer, sheet_name="TOP10_IPU_Error", index=False)
    top5_scenario_inn100.to_excel(writer, sheet_name="TOP5_Scenario_INN100", index=False)
    top5_ipu_inn100.to_excel(writer, sheet_name="TOP5_IPU_INN100", index=False)

    group_error_stats.to_excel(writer, sheet_name="Group_Error_Stats", index=False)
    group_default_stats.to_excel(writer, sheet_name="Group_Default_Stats_DR", index=False)
    group_default_inn_details.to_excel(writer, sheet_name="Group_Default_INN_DR", index=False)
    new_checks.to_excel(writer, sheet_name="Group_Checks", index=False)

print(f"Excel сохранен (расширенный): {EXCEL_PATH}")

In [ ]:
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    top10_scenario_error.to_excel(writer, sheet_name="TOP10_Scenario_Error", index=False)
    top10_ipu_error.to_excel(writer, sheet_name="TOP10_IPU_Error", index=False)
    top5_scenario_inn100.to_excel(writer, sheet_name="TOP5_Scenario_INN100", index=False)
    top5_ipu_inn100.to_excel(writer, sheet_name="TOP5_IPU_INN100", index=False)

print(f"Excel сохранен: {EXCEL_PATH}")

# Контрольные проверки
scenario_per_segment = top10_scenario_error.groupby("segment")["rank"].max().rename("max_rank")
ipu_per_segment = top10_ipu_error.groupby("segment")["rank"].max().rename("max_rank")
print("\nПроверка TOP-10 (max rank по сегментам):")
print("Scenario:\n", scenario_per_segment)
print("IPU:\n", ipu_per_segment)

if not top5_scenario_inn100.empty:
    chk1 = top5_scenario_inn100.groupby(["segment", "fp_num", "fp_type"])["inn"].nunique().max()
else:
    chk1 = 0
if not top5_ipu_inn100.empty:
    chk2 = top5_ipu_inn100.groupby(["segment", "fp_num", "fp_type"])["inn"].nunique().max()
else:
    chk2 = 0

print("\nПроверка лимита ИНН (должно быть <=100):")
print("Scenario max unique INN:", chk1)
print("IPU max unique INN:", chk2)

# Валидация новых метрик.
def validate_error_metrics(top_df: pd.DataFrame, label: str) -> None:
    if top_df.empty:
        print(f"\n{label}: данных для проверки нет")
        return

    in_range = ((top_df["error_share_pct"] >= 0) & (top_df["error_share_pct"] <= 100)).all()
    no_overflow = (top_df["error_count"] <= top_df["total_count"]).all()

    print(f"\n{label} — проверка error_share_pct в диапазоне [0,100]:", bool(in_range))
    print(f"{label} — проверка error_count <= total_count:", bool(no_overflow))

    if not in_range:
        display(top_df.loc[~((top_df["error_share_pct"] >= 0) & (top_df["error_share_pct"] <= 100))])
    if not no_overflow:
        display(top_df.loc[top_df["error_count"] > top_df["total_count"]])


validate_error_metrics(top10_scenario_error, "TOP10 Scenario")
validate_error_metrics(top10_ipu_error, "TOP10 IPU")